# Operation ShadowVault — Ransomware Incident Response Analysis

**Scenario:** A manufacturing company (Meridian Precision Mfg) detects unusual
activity on a Windows endpoint. Employees report slow systems and missing
files. This notebook walks through the SOC investigation of a multi-stage
attack: **Initial Access → Credential Theft → Lateral Movement → Data
Exfiltration Attempt → Ransomware Deployment**.

The underlying data is a synthetic log dataset (`data/raw/`) generated by
`src/log_generator.py`, combining normal business-day noise with an
embedded attack chain. The detection logic in `src/detectors/` is applied
below stage by stage, then correlated into a single incident timeline.

> This notebook analyzes and detects simulated attacker *behavior* in log
> data — it does not contain or execute any functional malware, exploit,
> or encryption code.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

try:
    import pandas as pd
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from utils import load_logs
from detectors import initial_access, credential_access, lateral_movement, exfiltration, ransomware
from correlation_engine import run_all_detectors, score_by_host, attack_chain_summary

pd.set_option("display.max_colwidth", 120)
sec, sysmon, fw, files = load_logs()

## 1. Raw Data Overview

Four log sources feed this investigation, mirroring what a real SOC
would pull from a SIEM: Windows Security events, Sysmon (EDR-style)
telemetry, perimeter firewall logs, and endpoint file-activity logs.

In [2]:
for name, df in [("Windows Security", sec), ("Sysmon", sysmon),
                 ("Firewall", fw), ("File Activity", files)]:
    print(f"{name:18s} {len(df):>4} rows   columns: {list(df.columns)}")

Windows Security     29 rows   columns: ['Timestamp', 'Hostname', 'EventID', 'EventDescription', 'Account', 'Domain', 'SourceIP', 'LogonType', 'Status']
Sysmon               90 rows   columns: ['Timestamp', 'Hostname', 'EventID', 'Image', 'CommandLine', 'ParentImage', 'ParentCommandLine', 'User', 'TargetImage', 'GrantedAccess', 'DestinationIP', 'DestinationPort', 'Protocol']
Firewall             40 rows   columns: ['Timestamp', 'SourceIP', 'SourcePort', 'DestinationIP', 'DestinationPort', 'Protocol', 'Action', 'BytesSent', 'BytesReceived', 'Direction']
File Activity        51 rows   columns: ['Timestamp', 'Hostname', 'Account', 'FilePath', 'Action', 'OriginalExtension', 'NewExtension', 'ProcessName', 'FileSizeBytes']


In [3]:
sysmon.head(5)

,Timestamp,Hostname,EventID,Image,CommandLine,ParentImage,ParentCommandLine,User,TargetImage,GrantedAccess,DestinationIP,DestinationPort,Protocol
0,2026-07-14 08:16:26,SRV-FILE-01,1,teams.exe,NaN,explorer.exe,NaN,svc_backup,NaN,NaN,NaN,NaN,NaN
1,2026-07-14 08:27:14,WKS-HR-03,1,chrome.exe,NaN,explorer.exe,NaN,r.patel,NaN,NaN,NaN,NaN,NaN
2,2026-07-14 08:31:43,WKS-ENG-12,1,chrome.exe,NaN,explorer.exe,NaN,t.oconnor,NaN,NaN,NaN,NaN,NaN
3,2026-07-14 08:35:09,WKS-IT-02,1,teams.exe,NaN,explorer.exe,NaN,j.alvarez,NaN,NaN,NaN,NaN,NaN
4,2026-07-14 08:40:29,WKS-IT-02,1,chrome.exe,NaN,explorer.exe,NaN,j.alvarez,NaN,NaN,NaN,NaN,NaN


## 2. Stage-by-Stage Detection

Each detector below is intentionally narrow — it looks for one specific,
well-known attacker behavior (a technique, in ATT&CK terms) rather than
trying to classify "suspicious" activity broadly. That's what keeps the
false-positive rate low against the ~200 rows of benign noise mixed into
the dataset.

### Stage 1 — Initial Access (T1566.001 / T1204.002 / T1059.001)

In [4]:
stage1 = pd.DataFrame(initial_access.detect(sysmon))
stage1

,Stage,Technique,MITRE_ID,Timestamp,Hostname,Account,Detail,Severity
0,1 - Initial Access,Office application spawned scripting engine with obfuscation flags,T1566.001 / T1204.002 / T1059.001,2026-07-14 09:16:12,WKS-FIN-07,s.jenkins,WINWORD.EXE -> powershell.exe :: powershell.exe -nop -w hidden -enc <base64-encoded-command>,Critical


### Stage 2 — Credential Theft (T1003.001)

In [5]:
stage2 = pd.DataFrame(credential_access.detect(sysmon, files))
stage2

,Stage,Technique,MITRE_ID,Timestamp,Hostname,Account,Detail,Severity
0,2 - Credential Theft,Process accessed LSASS memory with a credential-dumping access mask,T1003.001,2026-07-14 09:45:09,WKS-FIN-07,s.jenkins,svchost_upd.exe accessed lsass.exe (GrantedAccess=0x1010),Critical
1,2 - Credential Theft,Suspected LSASS memory dump artifact written to disk,T1003.001,2026-07-14 09:45:20,WKS-FIN-07,s.jenkins,"C:\Users\s.jenkins\AppData\Local\Temp\lsass_dump.tmp (61,204,112 bytes)",Critical


### Stage 3 — Lateral Movement (T1021.002 / T1569.002)

In [6]:
stage3 = pd.DataFrame(lateral_movement.detect(sec, sysmon))
stage3

,Stage,Technique,MITRE_ID,Timestamp,Hostname,Account,Detail,Severity
0,3 - Lateral Movement,Single account authenticated to multiple hosts in a short window,T1021.002,2026-07-14 10:55:34,SRV-FILE-01,j.alvarez,"j.alvarez logged into 3 hosts within 90 min: ['SRV-FILE-01', 'WKS-ENG-12', 'WKS-HR-03']",Critical
1,3 - Lateral Movement,Remote service installed (PsExec-style execution),T1569.002,2026-07-14 10:30:20,WKS-ENG-12,j.alvarez,Service installed on WKS-ENG-12 by j.alvarez from source 10.10.12.47,High
2,3 - Lateral Movement,Remote service installed (PsExec-style execution),T1569.002,2026-07-14 10:45:28,WKS-HR-03,j.alvarez,Service installed on WKS-HR-03 by j.alvarez from source 10.10.12.47,High
3,3 - Lateral Movement,Remote service installed (PsExec-style execution),T1569.002,2026-07-14 10:55:43,SRV-FILE-01,j.alvarez,Service installed on SRV-FILE-01 by j.alvarez from source 10.10.12.47,High
4,3 - Lateral Movement,Remote service installed (PsExec-style execution),T1569.002,2026-07-14 11:07:43,SRV-DC-01,j.alvarez,Service installed on SRV-DC-01 by j.alvarez from source 10.10.12.47,High


### Stage 4 — Data Exfiltration Attempt (T1560 / T1041)

In [7]:
stage4 = pd.DataFrame(exfiltration.detect(sysmon, fw))
stage4

,Stage,Technique,MITRE_ID,Timestamp,Hostname,Account,Detail,Severity
0,4 - Data Exfiltration Attempt,Archive utility used to stage data (likely pre-exfil compression),T1560,2026-07-14 12:31:00,SRV-FILE-01,j.alvarez,"7z.exe :: 7z.exe a -mx1 archive_backup.7z ""\\SRV-FILE-01\Shared\*""",High
1,4 - Data Exfiltration Attempt,Anomalously large outbound transfer to external host (NOT blocked),T1041,2026-07-14 12:44:00,10.10.5.10,n/a,"10.10.5.10 -> 203.0.113.77:443 (1,800,000,000 bytes, action=Allow)",Critical
2,4 - Data Exfiltration Attempt,Anomalously large outbound transfer to external host (blocked at perimeter),T1041,2026-07-14 12:51:30,10.10.5.10,n/a,"10.10.5.10 -> 203.0.113.77:443 (210,000,000 bytes, action=Blocked)",High


### Stage 5 — Ransomware Deployment (T1490 / T1486 / T1070.001)

In [8]:
stage5 = pd.DataFrame(ransomware.detect(sysmon, sec, files))
stage5

,Stage,Technique,MITRE_ID,Timestamp,Hostname,Account,Detail,Severity
0,5 - Ransomware Deployment,Volume shadow copies deleted (backup/recovery sabotage),T1490,2026-07-14 14:00:39,WKS-ENG-12,j.alvarez,vssadmin.exe delete shadows /all /quiet,Critical
1,5 - Ransomware Deployment,Volume shadow copies deleted (backup/recovery sabotage),T1490,2026-07-14 14:00:54,SRV-FILE-01,j.alvarez,vssadmin.exe delete shadows /all /quiet,Critical
2,5 - Ransomware Deployment,Volume shadow copies deleted (backup/recovery sabotage),T1490,2026-07-14 14:02:43,WKS-FIN-07,j.alvarez,vssadmin.exe delete shadows /all /quiet,Critical
3,5 - Ransomware Deployment,Volume shadow copies deleted (backup/recovery sabotage),T1490,2026-07-14 14:03:21,WKS-HR-03,j.alvarez,vssadmin.exe delete shadows /all /quiet,Critical
4,5 - Ransomware Deployment,Security audit log cleared (anti-forensics),T1070.001,2026-07-14 14:00:39,WKS-ENG-12,j.alvarez,1102 event log clear,High
5,5 - Ransomware Deployment,Security audit log cleared (anti-forensics),T1070.001,2026-07-14 14:00:54,SRV-FILE-01,j.alvarez,1102 event log clear,High
6,5 - Ransomware Deployment,Security audit log cleared (anti-forensics),T1070.001,2026-07-14 14:02:43,WKS-FIN-07,j.alvarez,1102 event log clear,High
7,5 - Ransomware Deployment,Security audit log cleared (anti-forensics),T1070.001,2026-07-14 14:03:21,WKS-HR-03,j.alvarez,1102 event log clear,High
8,5 - Ransomware Deployment,Mass file rename to a single unfamiliar extension (consistent with encryption),T1486,2026-07-14 14:26:00,SRV-FILE-01,j.alvarez,4 files renamed to *.shadowvault within 21s (process: encryptor.exe),Critical
9,5 - Ransomware Deployment,Mass file rename to a single unfamiliar extension (consistent with encryption),T1486,2026-07-14 14:14:00,WKS-ENG-12,j.alvarez,4 files renamed to *.shadowvault within 21s (process: encryptor.exe),Critical


## 3. Correlated Incident Timeline

`correlation_engine.py` merges every stage's alerts into one chronological
timeline and scores hosts by cumulative severity, so an analyst can
immediately see the full kill chain and which assets were hit hardest.

In [9]:
timeline = run_all_detectors()
risk = score_by_host(timeline)
summary = attack_chain_summary(timeline)

print(f"{len(timeline)} correlated alerts across {timeline['Stage'].nunique()} stages")
summary

27 correlated alerts across 5 stages


,Stage,Alerts,First_Seen,Last_Seen
0,1 - Initial Access,1,2026-07-14 09:16:12,2026-07-14 09:16:12
1,2 - Credential Theft,2,2026-07-14 09:45:09,2026-07-14 09:45:20
2,3 - Lateral Movement,5,2026-07-14 10:30:20,2026-07-14 11:07:43
3,4 - Data Exfiltration Attempt,3,2026-07-14 12:31:00,2026-07-14 12:51:30
4,5 - Ransomware Deployment,16,2026-07-14 14:00:39,2026-07-14 14:30:45


## 4. Visualizations

**Attack timeline** — every alert plotted chronologically, colored by stage.

In [10]:
fig = px.scatter(
    timeline, x="Timestamp", y="Stage", color="Stage", symbol="Severity",
    hover_data=["Hostname", "Account", "Technique", "MITRE_ID"],
    title="Operation ShadowVault — Attack Chain Timeline",
    height=450,
)
fig.update_traces(marker=dict(size=13, line=dict(width=1, color="white")))
fig.update_layout(showlegend=True, xaxis_title="Time", yaxis_title="Attack Stage")
fig.show()

**Host risk ranking** — cumulative severity-weighted score per asset.

In [11]:
fig = px.bar(
    risk, x="Hostname", y="RiskScore", color="RiskScore",
    color_continuous_scale="Reds", title="Host Risk Score (severity-weighted alert count)",
    height=400,
)
fig.show()

**Lateral movement path** — who the stolen admin account (`j.alvarez`) touched, and when.

In [12]:
lat_events = sec[(sec["EventID"] == 4624) & (sec["LogonType"] == 3) &
                 (sec["Account"] == "j.alvarez")].sort_values("Timestamp")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lat_events["Timestamp"], y=lat_events["Hostname"],
    mode="markers+lines+text", text=lat_events["Hostname"], textposition="top center",
    marker=dict(size=14, color="crimson"), line=dict(color="lightgrey", dash="dot"),
))
fig.update_layout(title="Lateral Movement Path — j.alvarez (stolen IT admin credentials)",
                   xaxis_title="Time", yaxis_title="Host", height=400)
fig.show()

**Exfiltration spike** — outbound bytes over time, highlighting the anomalous transfer.

In [13]:
fw_sorted = fw.sort_values("Timestamp")
fig = px.bar(
    fw_sorted, x="Timestamp", y="BytesSent", color="Action",
    color_discrete_map={"Allow": "orange", "Blocked": "green"},
    title="Outbound Traffic Volume Over Time (flagged transfer highlighted)",
    height=400,
)
fig.show()

## 5. Generated Incident Report

`report_generator.py` renders the same correlated data into a formatted
Markdown incident report (`reports/incident_report.md`) — the kind of
artifact you'd actually hand to a manager or attach to a resume writeup.

In [14]:
from report_generator import build_report
report_md = build_report(timeline, risk, summary)
print(report_md[:1500] + "\n...\n[full report continues in reports/incident_report.md]")

# Incident Report: Operation ShadowVault
**Generated:** 2026-07-25 20:36  
**Classification:** Simulated Incident (Training Exercise)  
**Organization:** Meridian Precision Manufacturing (fictional)  
**Incident Date:** 2026-07-14

## Executive Summary
On 2026-07-14, an employee in Accounts Payable opened a malicious attachment delivered via email, triggering a five-stage intrusion that progressed from initial access to full ransomware deployment in approximately **5.2 hours**. The attacker dumped credentials from a compromised finance workstation, used a stolen IT administrator account to move laterally across **6 hosts**, staged and attempted to exfiltrate proprietary data to external infrastructure, and ultimately deployed ransomware that deleted shadow copy backups and encrypted files across four endpoints and the file server. This report reconstructs the full attack chain from correlated log data.

## Attack Chain Overview
| Stage | Alerts | First Observed | Last Observed |
|---|-

## Conclusion

This simulation walks through a complete ransomware kill chain — phishing,
credential theft, lateral movement, attempted exfiltration, and encryption
— purely through log correlation and MITRE ATT&CK-mapped detection logic,
with no manual "we know where the attack is" shortcuts baked into the
detectors. That structure (independent, technique-scoped detectors feeding
a correlation layer) mirrors how real detection engineering teams build
and reason about SOC content.

**Possible extensions:**
- Add a sixth detector for C2 beaconing (regular-interval outbound connections).
- Swap the severity-weighted risk score for a proper anomaly-detection model.
- Feed `data/processed/incident_timeline.csv` into a SIEM-style dashboard (e.g. Streamlit).